# Consolidating Fingerprint Prediction Results

This notebook aggregates the evaluation results produced by different fingerprint prediction models.

Each experiment saves a `test_performance.json` file containing evaluation metrics computed on the test split. In particular, we extract the **Jaccard similarity** between predicted and ground-truth molecular fingerprints.

The workflow of this notebook is:

1. Load evaluation results from the `results/FP_prediction` directory.
2. Iterate over all:
   - models
   - datasets
   - training checkpoints / splits
3. Extract the **Jaccard score** stored in each `test_performance.json`.
4. Consolidate all results into a single table for easier comparison.

The Jaccard similarity between two binary fingerprints \(A\) and \(B\) is defined as:

$$
J(A, B) = \frac{|A \cap B|}{|A \cup B|}
$$

Finally, the results are formatted into a pandas dataframe to allow easy inspection of model performance across datasets.

Import libraries

In [1]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
from utils import load_json

Settings

In [2]:
results_folder = Path("../results/FP_prediction")

Consolidate the results

In [3]:
all_test_performances = []

for model in os.listdir(results_folder):

    dataset_folder = results_folder / model
    
    for dataset in os.listdir(dataset_folder):

        ckpt_list = dataset_folder / dataset 
        
        for ckpt in os.listdir(ckpt_list):
            
            split = ckpt.split("_")[-1]
            test_results = load_json(ckpt_list / ckpt / "test_performance.json")

            all_test_performances.append([model, dataset, split, round(test_results["jaccard"],3)])

Format the table

In [7]:
test_performance_df = pd.DataFrame(all_test_performances, columns=["model", "dataset", "split", "jaccard"])

for dataset in ["NPLIB1", "massspecgym"]:
    print(test_performance_df.query("dataset == @dataset"))
    print()

             model dataset     split  jaccard
3   MS_transformer  NPLIB1    random    0.413
4   MS_transformer  NPLIB1        LS    0.187
5   MS_transformer  NPLIB1  scaffold    0.230
9   CF_transformer  NPLIB1    random    0.464
10  CF_transformer  NPLIB1        LS    0.186
11  CF_transformer  NPLIB1  scaffold    0.231
15      binned_MLP  NPLIB1    random    0.444
16      binned_MLP  NPLIB1        LS    0.184
17      binned_MLP  NPLIB1  scaffold    0.237
18            mist  NPLIB1  scaffold    0.241
19            mist  NPLIB1        LS    0.231
20            mist  NPLIB1    random    0.547

             model      dataset     split  jaccard
0   MS_transformer  massspecgym        LS    0.191
1   MS_transformer  massspecgym    random    0.668
2   MS_transformer  massspecgym  scaffold    0.228
6   CF_transformer  massspecgym    random    0.638
7   CF_transformer  massspecgym        LS    0.194
8   CF_transformer  massspecgym  scaffold    0.254
12      binned_MLP  massspecgym    random   